### Scaling & Normalization

Loading data

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler,RobustScaler,MinMaxScaler
import pickle

In [2]:
df = pd.read_csv("../outputs/outliers_handled.csv")
print("Loaded shape:", df.shape)

Loaded shape: (149233, 11)


Encoding Check

In [3]:
print("Data types:")
print(df.dtypes)

categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

print(f"\nCategorical columns: {categorical_cols if categorical_cols else 'None'}")
print(f"Numerical columns  : {numerical_cols}")

print("\nAll features are numerical — no categorical encoding needed.")
print("Target is already binary (0/1).")

Data types:
Target                    int64
RevolvingUtilization    float64
Age                       int64
Times30_59Late            int64
DebtRatio               float64
MonthlyIncome           float64
OpenCreditLines           int64
Times90Late               int64
RealEstateLines           int64
Times60_89Late            int64
Dependents              float64
dtype: object

Categorical columns: None
Numerical columns  : ['Target', 'RevolvingUtilization', 'Age', 'Times30_59Late', 'DebtRatio', 'MonthlyIncome', 'OpenCreditLines', 'Times90Late', 'RealEstateLines', 'Times60_89Late', 'Dependents']

All features are numerical — no categorical encoding needed.
Target is already binary (0/1).


Scaler Comparison

In [4]:
feature_cols = [col for col in df.columns if col != "Target"]
X = df[feature_cols]
y = df["Target"]

print(f"Features: {len(feature_cols)}")
col = "MonthlyIncome"
print(f"\nBefore scaling ({col}):")
print(f"  Min={X[col].min():.0f}, Max={X[col].max():.0f}, Mean={X[col].mean():.0f}, Std={X[col].std():.0f}")
print(f"\n  {'Scaler':<20} {'Min':>8} {'Max':>8} {'Mean':>8} {'Std':>8}")

print("  " + "-" * 57)
for name, scaler in [("StandardScaler", StandardScaler()),
                      ("MinMaxScaler", MinMaxScaler()),
                      ("RobustScaler", RobustScaler())]:
    scaled = pd.DataFrame(scaler.fit_transform(X), columns=feature_cols)
    print(f"  {name:<20} {scaled[col].min():>8.2f} {scaled[col].max():>8.2f} "
          f"{scaled[col].mean():>8.2f} {scaled[col].std():>8.2f}")
print("\nUsing RobustScaler (uses median/IQR, handles remaining skew better)")

Features: 10

Before scaling (MonthlyIncome):
  Min=0, Max=23098, Mean=6150, Std=3847

  Scaler                    Min      Max     Mean      Std
  ---------------------------------------------------------
  StandardScaler          -1.60     4.41    -0.00     1.00
  MinMaxScaler             0.00     1.00     0.27     0.17
  RobustScaler            -1.54     5.04     0.21     1.10

Using RobustScaler (uses median/IQR, handles remaining skew better)


Apply RobustScaler

In [5]:
scaler = RobustScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=feature_cols)

print("Before vs After:")
print(f"  {'Feature':<28} {'Before Min':>10} {'Before Max':>10} {'After Min':>10} {'After Max':>10}")

print("  " + "-" * 72)
for col in feature_cols:
    print(f"  {col:<28} {X[col].min():>10.2f} {X[col].max():>10.2f} "
          f"{X_scaled[col].min():>10.2f} {X_scaled[col].max():>10.2f}")

Before vs After:
  Feature                      Before Min Before Max  After Min  After Max
  ------------------------------------------------------------------------
  RevolvingUtilization               0.00       1.09      -0.29       1.79
  Age                               21.00     109.00      -1.41       2.59
  Times30_59Late                     0.00      20.00       0.00      20.00
  DebtRatio                          0.00    4988.04      -0.53    7135.52
  MonthlyIncome                      0.00   23097.76      -1.54       5.04
  OpenCreditLines                    0.00      24.00      -1.33       2.67
  Times90Late                        0.00      20.00       0.00      20.00
  RealEstateLines                    0.00       4.00      -0.50       1.50
  Times60_89Late                     0.00      20.00       0.00      20.00
  Dependents                         0.00      20.00       0.00      20.00


Saving the csv file

In [8]:
df_scaled = pd.concat([X_scaled, y.reset_index(drop=True)], axis=1)
df_scaled.to_csv("../outputs/scaled_dataset.csv", index=False)

print(f"Saved: outputs/scaled_dataset.csv ({df_scaled.shape[0]:,} rows)")

with open("../outputs/robust_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
print("Saved: outputs/robust_scaler.pkl")

Saved: outputs/scaled_dataset.csv (149,233 rows)
Saved: outputs/robust_scaler.pkl
